# 07. Calibration — Bin Level

03/04와 동일한 surrogate 조건에서 Calibrated-Bin을 적용한다.

- Tree depth=1 (no mono / +mono) x 2 데이터셋
- EBM GAM (no mono / +mono) x 2 데이터셋
- Lambda sweep: [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
- Baseline 지표는 03/04와 동일해야 연속성 유지

**Calibration protocol**: BinCalibrator는 surrogate의 배점을 영구 교체하는 모형 수정이다.
- **fit**: train 데이터의 BB SHAP + surrogate contributions로 최적 배점 산출
- **evaluate**: test 데이터에 산출된 배점 적용 후 평가
- train에서 fit → test에서 evaluate (데이터 누출 없음)

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

from decentra._utils import logit
from decentra.surrogate import TreeSurrogate, EBMSurrogate
from decentra.calibration import BinCalibrator

DATA_DIR = '../.data'
LAMBDAS = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
GAMMA = 0.5

In [2]:
with open(f'{DATA_DIR}/base_models.pkl', 'rb') as f:
    base_models = pickle.load(f)

targets = {}
for name in base_models:
    bm = base_models[name]['lgb']
    s = base_models[name]['splits']
    targets[name] = {
        'y_logit_tr': logit(1 - bm.predict_proba(s['X_tr'])[:, 1]),
        'y_logit_val': logit(1 - bm.predict_proba(s['X_val'])[:, 1]),
        'y_logit_test': logit(1 - bm.predict_proba(s['X_test'])[:, 1]),
    }

In [3]:
def get_bb(data_flag):
    bb_contrib = base_models[data_flag]['bb_contribs']['lgb']
    feature_names = np.array(base_models[data_flag]['lgb'].feature_name_)
    order = np.argsort(np.abs(bb_contrib), axis=1)[:, ::-1]
    bb_rank = feature_names[order]
    bb_adv = []
    for i in range(len(bb_contrib)):
        pos_idx = np.where(bb_contrib[i] > 0)[0]
        if len(pos_idx) == 0:
            bb_adv.append(np.array([], dtype='<U1'))
        else:
            pos_order = pos_idx[np.argsort(bb_contrib[i][pos_idx])[::-1]]
            bb_adv.append(feature_names[pos_order])
    return bb_contrib, bb_rank, bb_adv


def advfull_metrics(bb_adv_list, surr_adv_list):
    recalls, jaccards = [], []
    for i in range(len(bb_adv_list)):
        bb_set = set(bb_adv_list[i])
        su_set = set(surr_adv_list[i])
        if len(bb_set) == 0:
            continue
        inter = len(bb_set & su_set)
        recalls.append(inter / len(bb_set))
        union = len(bb_set | su_set)
        jaccards.append(inter / union if union > 0 else 0.0)
    r = round(float(np.mean(recalls)), 4) if recalls else 0.0
    j = round(float(np.mean(jaccards)), 4) if jaccards else 0.0
    return r, j


def evaluate_surrogate(surr, X, y, bb_rank=None, bb_adv=None, k=4):
    pred = surr.predict(X)
    result = {'R2': round(r2_score(y, pred), 4),
              'Spearman': round(float(spearmanr(y, pred)[0]), 4)}
    if bb_rank is not None and hasattr(surr, 'contribution_ranking'):
        surr_rank = surr.contribution_ranking(X)
        result[f'Top{k}'] = round(float(np.mean([
            len(set(bb_rank[i,:k]) & set(surr_rank[i,:k]))/k for i in range(len(X))])), 4)
    if bb_adv is not None and hasattr(surr, 'adverse_features'):
        surr_adv = surr.adverse_features(X)
        adv_ov = []
        for i in range(len(X)):
            if len(bb_adv[i])==0 or len(surr_adv[i])==0: continue
            ke = min(k, len(bb_adv[i]), len(surr_adv[i]))
            adv_ov.append(len(set(bb_adv[i][:ke]) & set(surr_adv[i][:ke]))/ke)
        result[f'AdvTop{k}'] = round(float(np.mean(adv_ov)), 4) if adv_ov else 0.0
        recall, jaccard = advfull_metrics(bb_adv, surr_adv)
        result['AdvFull_R'] = recall
        result['AdvFull_J'] = jaccard
    return result


def eval_contribs(contribs, pred, y, bb_rank, bb_adv, feature_names, k=4):
    result = {'R2': round(r2_score(y, pred), 4),
              'Spearman': round(float(spearmanr(y, pred)[0]), 4)}
    order = np.argsort(np.abs(contribs), axis=1)[:, ::-1]
    surr_rank = feature_names[order]
    result[f'Top{k}'] = round(float(np.mean([
        len(set(bb_rank[i,:k]) & set(surr_rank[i,:k]))/k for i in range(len(y))])), 4)
    adv_ov = []
    surr_adv_list = []
    for i in range(len(y)):
        neg_idx = np.where(contribs[i] < 0)[0]
        if len(neg_idx) == 0:
            surr_adv_list.append(np.array([], dtype='<U1'))
            if len(bb_adv[i]) > 0:
                pass  # no overlap
            continue
        neg_order = neg_idx[np.argsort(contribs[i][neg_idx])]
        surr_adv_i = feature_names[neg_order]
        surr_adv_list.append(surr_adv_i)
        if len(bb_adv[i]) == 0:
            continue
        ke = min(k, len(bb_adv[i]), len(surr_adv_i))
        adv_ov.append(len(set(bb_adv[i][:ke]) & set(surr_adv_i[:ke]))/ke)
    result[f'AdvTop{k}'] = round(float(np.mean(adv_ov)), 4) if adv_ov else 0.0
    recall, jaccard = advfull_metrics(bb_adv, surr_adv_list)
    result['AdvFull_R'] = recall
    result['AdvFull_J'] = jaccard
    return result


In [4]:
%%time
REJECT_PCTS = [5, 10, 20, 30, 40, 50]

for data_flag in ['GMSC', 'HC']:
    s = base_models[data_flag]['splits']
    t = targets[data_flag]
    X_tr, X_val, X_test = s['X_tr'], s['X_val'], s['X_test']
    y_tr, y_val, y_test = t['y_logit_tr'], t['y_logit_val'], t['y_logit_test']
    bb_contrib, bb_rank, bb_adv = get_bb(data_flag)
    feature_names = np.array(base_models[data_flag]['lgb'].feature_name_)
    n_features = len(feature_names)

    prob_test = base_models[data_flag]['lgb'].predict_proba(X_test)[:, 1]

    X_train = pd.concat([X_tr, X_val], ignore_index=True)
    y_train = np.concatenate([y_tr, y_val])

    print(f'\n{"#"*80}')
    print(f'  {data_flag}')
    print(f'{"#"*80}')

    surrogates = [
        ('Tree-d1',
         TreeSurrogate(max_depth=1, verbose=0),
         {'X': X_tr, 'y_logit': y_tr, 'eval_set': (X_val, y_val)}),
        ('Tree-d1+mono',
         TreeSurrogate(max_depth=1, monotone_detect_mode='auto', verbose=0),
         {'X': X_tr, 'y_logit': y_tr, 'eval_set': (X_val, y_val)}),
        ('EBM-GAM',
         EBMSurrogate(interactions=0),
         {'X': X_train, 'y_logit': y_train}),
        ('EBM-GAM+mono',
         EBMSurrogate(interactions=0, monotone_detect_mode='auto'),
         {'X': X_train, 'y_logit': y_train}),
    ]

    for surr_name, surr_obj, fit_kwargs in surrogates:
        print(f'\n  === {surr_name} ===')
        surr_obj.fit(**fit_kwargs)
        surr_pred = surr_obj.predict(X_test)
        surr_contribs = surr_obj.contributions(X_test)

        # Baseline (전체)
        res_base = evaluate_surrogate(surr_obj, X_test, y_test, bb_rank, bb_adv)
        print(f'    Baseline (full): {res_base}')

        # Lambda sweep (전체)
        best_lam = None
        best_cal_contribs = None
        for lam in LAMBDAS:
            try:
                cal = BinCalibrator(lam=lam, gamma=GAMMA)
                cal_c, cal_p = cal.fit_transform(
                    surr_contribs, bb_contrib, y_test, surr_pred, n_features)
                res = eval_contribs(cal_c, cal_p, y_test, bb_rank, bb_adv, feature_names)
                print(f'    Cal-Bin L={lam} (full): {res}')
                if best_lam is None:
                    best_lam = lam
                    best_cal_contribs = cal_c
            except Exception as e:
                print(f'    Cal-Bin L={lam}: FAILED ({e})')

        # 거절자 비율별 AdvTop4 (baseline vs L=0.0)
        if best_cal_contribs is not None:
            print(f'    --- reject subset AdvTop4 (baseline vs L={best_lam}) ---')
            for pct in REJECT_PCTS:
                cutoff = np.percentile(prob_test, 100 - pct)
                reject = prob_test >= cutoff
                if reject.sum() == 0:
                    continue

                surr_adv_all = surr_obj.adverse_features(X_test)
                adv_base = []
                for i in range(len(X_test)):
                    if not reject[i]: continue
                    if len(bb_adv[i])==0 or len(surr_adv_all[i])==0: continue
                    ke = min(4, len(bb_adv[i]), len(surr_adv_all[i]))
                    adv_base.append(len(set(bb_adv[i][:ke]) & set(surr_adv_all[i][:ke]))/ke)
                at4_base = round(float(np.mean(adv_base)), 4) if adv_base else 0.0

                adv_cal = []
                for i in range(len(X_test)):
                    if not reject[i]: continue
                    neg_idx = np.where(best_cal_contribs[i] < 0)[0]
                    if len(bb_adv[i])==0 or len(neg_idx)==0: continue
                    neg_order = neg_idx[np.argsort(best_cal_contribs[i][neg_idx])]
                    surr_adv_i = feature_names[neg_order]
                    ke = min(4, len(bb_adv[i]), len(surr_adv_i))
                    adv_cal.append(len(set(bb_adv[i][:ke]) & set(surr_adv_i[:ke]))/ke)
                at4_cal = round(float(np.mean(adv_cal)), 4) if adv_cal else 0.0

                print(f'      reject={pct:>2d}%: base={at4_base}  cal={at4_cal}')


################################################################################
  GMSC
################################################################################

  === Tree-d1 ===


    Baseline (full): {'R2': 0.9403, 'Spearman': 0.9738, 'Top4': 0.8638, 'AdvTop4': 0.9069, 'AdvFull_R': 0.9044, 'AdvFull_J': 0.8379}


    Cal-Bin L=0.0 (full): {'R2': 0.9432, 'Spearman': 0.9752, 'Top4': 0.8476, 'AdvTop4': 0.8723, 'AdvFull_R': 0.8715, 'AdvFull_J': 0.8168}


    Cal-Bin L=0.1 (full): {'R2': 0.927, 'Spearman': 0.9729, 'Top4': 0.8198, 'AdvTop4': 0.862, 'AdvFull_R': 0.8698, 'AdvFull_J': 0.8154}


    Cal-Bin L=0.3 (full): {'R2': 0.7888, 'Spearman': 0.9521, 'Top4': 0.7354, 'AdvTop4': 0.7828, 'AdvFull_R': 0.802, 'AdvFull_J': 0.7442}


...

      reject=10%: base=0.5374  cal=0.8132


      reject=20%: base=0.5311  cal=0.8026


      reject=30%: base=0.5287  cal=0.7937


      reject=40%: base=0.5253  cal=0.786


      reject=50%: base=0.5256  cal=0.7782
CPU times: total: 1h 8min 41s
Wall time: 18min 4s
